# Huấn luyện, Hiệu chỉnh Ngưỡng và Đánh giá CAE trên Google Colab

**Đề tài:** Xây dựng và đánh giá mạng Convolutional Autoencoder trong phát hiện và khoanh vùng bất thường bề mặt sản phẩm công nghiệp.

> **Quy trình hoạt động:** 
> - Logic thuật toán được quản lý trong các module `.py` trên GitHub.
> - Notebook này vừa đóng vai trò **bảng điều khiển (launcher)** gọi các lệnh thực thi GPU, vừa là **bảng hiển thị trực quan (dashboard)** để bạn xem đồ thị loss, bảng điểm số và các bức ảnh phát hiện/khoanh vùng lỗi ngay trên trình duyệt.

--- 
## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

--- 
## 2. Đồng bộ mã nguồn từ GitHub & Cài đặt thư viện

*(Nếu là lần đầu chạy: clone repo. Các lần sau: chỉ cần `git pull` để nhận code mới nhất từ máy cá nhân)*

In [ ]:
import os

# THAY THẾ URL REPOSITORY CỦA BẠN TẠI ĐÂY:
REPO_URL = "https://github.com/<your_username>/industrial-anomaly-cae.git"
REPO_DIR = "/content/industrial-anomaly-cae"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

!pip install -r requirements.txt -q
print("[*] Đã đồng bộ mã nguồn và cài đặt thư viện thành công!")

--- 
## 3. Chuẩn bị Dữ liệu & Xem trước Ảnh Mẫu (Data Preview)

Sao chép danh mục `bottle` từ Google Drive sang ổ đĩa tạm `/content/` để nạp dữ liệu nhanh nhất.

In [ ]:
# Tạo thư mục tạm trên Colab runtime
!mkdir -p /content/cae_data/mvtec_anomaly_detection

DRIVE_DATASET_PATH = "/content/drive/MyDrive/CAE_Project/datasets/mvtec_anomaly_detection/bottle"
LOCAL_DATASET_PATH = "/content/cae_data/mvtec_anomaly_detection/bottle"

# Sao chép danh mục sang /content/
!cp -r "{DRIVE_DATASET_PATH}" "/content/cae_data/mvtec_anomaly_detection/"
print("[*] Dữ liệu đã sẵn sàng tại:", LOCAL_DATASET_PATH)

In [ ]:
# Trực quan hóa: Xem trước ảnh bình thường và ảnh lỗi trong tập dữ liệu
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

base_path = Path(LOCAL_DATASET_PATH)
train_good = sorted(list((base_path / "train" / "good").glob("*.png")))
test_defect = sorted(list((base_path / "test" / "broken_large").glob("*.png")))
gt_defect = sorted(list((base_path / "ground_truth" / "broken_large").glob("*.png")))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
if train_good:
    axes[0].imshow(Image.open(train_good[0]))
    axes[0].set_title(f"Train Normal (Good)\n{train_good[0].name}")
axes[0].axis("off")

if test_defect:
    axes[1].imshow(Image.open(test_defect[0]))
    axes[1].set_title(f"Test Anomaly (Defect)\n{test_defect[0].name}")
axes[1].axis("off")

if gt_defect:
    axes[2].imshow(Image.open(gt_defect[0]), cmap="gray")
    axes[2].set_title(f"Ground Truth Mask\n{gt_defect[0].name}")
axes[2].axis("off")

plt.suptitle("Kiểm tra tính toàn vẹn của Dữ liệu MVTec AD Bottle", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

--- 
## 4. Kiểm tra Kiến trúc Mạng Convolutional Autoencoder (CAE)

In [ ]:
import torch
from src.models.cae import ConvolutionalAutoencoder

model = ConvolutionalAutoencoder(in_channels=3, base_channels=32, output_activation="sigmoid")
dummy_input = torch.randn(1, 3, 256, 256)
dummy_output = model(dummy_input)

print("=== KIẾN TRÚC MẠNG CONVOLUTIONAL AUTOENCODER ===")
print(f"- Input Shape:        {list(dummy_input.shape)}")
print(f"- Reconstructed Shape: {list(dummy_output.shape)}")
print(f"- Tổng số tham số huấn luyện: {model.count_parameters():,} parameters")
print("\nCấu trúc chi tiết:")
print(model)

--- 
## 5. Huấn luyện Mô hình CAE (train.py)

Quá trình huấn luyện chỉ sử dụng ảnh bình thường (`train/good`). Checkpoint tốt nhất (`best_model.pth`) và log loss sẽ được tự động lưu vào Google Drive.

In [ ]:
!python train.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

In [ ]:
# Hiển thị Biểu đồ Loss Curve trực tiếp trên Notebook
from IPython.display import Image, display
import pandas as pd

exp_dir = "/content/drive/MyDrive/CAE_Project/outputs/E1_bottle_mse_seed42"
loss_img_path = f"{exp_dir}/plots/bottle_loss_curve.png"
log_csv_path = f"{exp_dir}/logs/bottle_train.csv"

if os.path.exists(loss_img_path):
    print("[*] Đồ thị huấn luyện (Loss Curve):")
    display(Image(filename=loss_img_path))

if os.path.exists(log_csv_path):
    df_log = pd.read_csv(log_csv_path)
    print("\n[*] 5 Epochs cuối cùng:")
    display(df_log.tail(5))
    print(f"[*] Best Val Loss: {df_log['val_loss'].min():.6f} (Epoch {df_log.loc[df_log['val_loss'].idxmin(), 'epoch']})")

--- 
## 6. Hiệu chỉnh Ngưỡng trên Tập Validation (calibrate.py)

Xác định `image_threshold` và `pixel_threshold` hoàn toàn từ phân phối sai số của ảnh bình thường trên tập validation (chống rò rỉ dữ liệu test).

In [ ]:
!python calibrate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

In [ ]:
# Hiển thị thông số ngưỡng đã hiệu chỉnh
import json

thresh_file = f"{exp_dir}/thresholds/bottle_thresholds.json"
if os.path.exists(thresh_file):
    with open(thresh_file, 'r') as f:
        tdata = json.load(f)
    print("=== KẾT QUẢ HIỆU CHỈNH NGƯỠNG (CALIBRATION) ===")
    print(f"- Image Threshold: {tdata['image_threshold']:.6f} (Rule: {tdata['image_threshold_rule']})")
    print(f"- Pixel Threshold: {tdata['pixel_threshold']:.6f} (Rule: {tdata['pixel_threshold_rule']})")
    print(f"- Validation Stats: Min={tdata['validation_stats']['min']:.6f}, Mean={tdata['validation_stats']['mean']:.6f}, Max={tdata['validation_stats']['max']:.6f}")
    print(f"- Normal False Positive Rate trên Val: {tdata['validation_stats']['val_fp_rate']*100:.2f}%")

--- 
## 7. Đánh giá Độc lập trên Tập Test (evaluate.py)

Đánh giá toàn diện trên toàn bộ ảnh test (cả bình thường lẫn tất cả các loại khuyết tật: `broken_large`, `broken_small`, `contamination`).

In [ ]:
!python evaluate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42 \
  --max-vis-samples 4

In [ ]:
# Hiển thị Bảng Điểm Số và Đồ Thị Đánh Giá Cuối Cùng
import json
from IPython.display import Image, display

metrics_file = f"{exp_dir}/metrics/bottle_metrics.json"
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        mdata = json.load(f)
    print("==================================================")
    print("           BẢNG ĐIỂM ĐÁNH GIÁ TRÊN TẬP TEST       ")
    print("==================================================")
    print(f"* Image-level AUROC: {mdata['image_level']['auroc']:.4f}")
    print(f"* Image-level F1:    {mdata['image_level']['f1']:.4f}")
    print(f"* Precision:         {mdata['image_level']['precision']:.4f}")
    print(f"* Recall:            {mdata['image_level']['recall']:.4f}")
    print(f"* Confusion Matrix:  {mdata['image_level']['confusion_matrix']}")
    print("--------------------------------------------------")
    print(f"* Pixel-level AUROC: {mdata['pixel_level']['pixel_auroc']:.4f}")
    print(f"* Pixel-level IoU:   {mdata['pixel_level']['pixel_iou']:.4f}")
    print(f"* Pixel-level Dice:  {mdata['pixel_level']['pixel_dice']:.4f}")
    print("==================================================")

roc_img = f"{exp_dir}/plots/bottle_image_roc.png"
cm_img = f"{exp_dir}/plots/bottle_confusion_matrix.png"

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if os.path.exists(roc_img):
    axes[0].imshow(plt.imread(roc_img))
    axes[0].set_title("Image-level ROC Curve")
axes[0].axis("off")

if os.path.exists(cm_img):
    axes[1].imshow(plt.imread(cm_img))
    axes[1].set_title("Confusion Matrix")
axes[1].axis("off")

plt.tight_layout()
plt.show()

--- 
## 8. Xem Trực Quan Các Trường Hợp Khoanh Vùng Lỗi (Qualitative Results)

Hiển thị chi tiết 6 khung hình phân tích: **Ảnh gốc $\rightarrow$ Ảnh tái tạo $\rightarrow$ Anomaly Heatmap $\rightarrow$ Mask dự đoán $\rightarrow$ Vùng khoanh lỗi Overlay $\rightarrow$ Ground Truth Mask**

In [ ]:
from pathlib import Path
from IPython.display import Image, display

vis_dir = Path(exp_dir) / "visualizations" / "bottle"
anomaly_samples = sorted(list((vis_dir / "anomaly").glob("*.png")))
normal_samples = sorted(list((vis_dir / "normal").glob("*.png")))

print("=== VÍ DỤ PHÁT HIỆN SẢN PHẨM BỊ LỖI (TRUE POSITIVE) ===")
for s in anomaly_samples[:2]:
    print(f"Sample: {s.name}")
    display(Image(filename=str(s)))

print("\n=== VÍ DỤ PHÁT HIỆN SẢN PHẨM BÌNH THƯỜNG (TRUE NEGATIVE) ===")
for s in normal_samples[:1]:
    print(f"Sample: {s.name}")
    display(Image(filename=str(s)))